# CS 179 Project

Group 47: Ryan Noel Buenaventura, Jakob Paul Groh, Jason Hong Liang Wong

Compare naive bayes vs tree augmented naive bayes (tan) on the uci student performance dataset. both are bayesian networks.

# Resources
- Dataset: https://archive.ics.uci.edu/dataset/320/student+performance

In [77]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from pgmpy.estimators import TreeSearch
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator

# Load Data

In [78]:
df = pd.read_csv('student-mat.csv', sep=';')

# pass if final grade >= 10
df['pass'] = (df['G3'] >= 10).astype(int)

# reduce down from ~35 to 4
df['absences']  = pd.cut(df['absences'], bins=[-1, 0, 6, 25, 100], labels=['none', 'moderate', 'high', 'severe'])

# reduce down from 7 to 2
df['age'] = pd.cut(df['age'], bins=[14, 18, 22], labels=['teen', 'adult'])

feature_cols = [c for c in df.columns if c not in ['G1', 'G2', 'G3', 'pass']]
X = df[feature_cols]
y = df['pass']

# consolidate types
X = X.astype(str)

print(f'students: {len(df)}, pass: {y.sum()}, fail: {(y==0).sum()}')

students: 395, pass: 265, fail: 130


In [79]:
# split into 80/20 train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=.2, random_state=123, stratify=y
)

print(f'train: {len(X_train)}, test: {len(X_test)}')

train: 316, test: 79


In [80]:
def evaluate_model(fitted_model, X, y):
    pred = fitted_model.predict(X)
    print(f"Accuracy: {accuracy_score(y.astype(str), pred['pass']):.5f}")
    print(classification_report(y.astype(str), pred['pass']))

# Model 1: Naive Bayes

In [81]:
enc = OrdinalEncoder()

X_train_sk = enc.fit_transform(X_train)
X_test_sk = enc.transform(X_test)

nb = CategoricalNB()
nb.fit(X_train_sk, y_train)
nb_acc = accuracy_score(y_test, nb.predict(X_test_sk))

print(f'naive bayes accuracy: {nb_acc:.5f}')
print(classification_report(y_test, nb.predict(X_test_sk), target_names=['fail', 'pass']))

naive bayes accuracy: 0.69620
              precision    recall  f1-score   support

        fail       0.56      0.35      0.43        26
        pass       0.73      0.87      0.79        53

    accuracy                           0.70        79
   macro avg       0.65      0.61      0.61        79
weighted avg       0.67      0.70      0.67        79



In [84]:
train_df = X_train.copy()
train_df['pass'] = y_train.values.astype(str)

features = [col for col in train_df.columns if col != 'pass']
edges = [('pass', feature) for feature in features]

naive_model = DiscreteBayesianNetwork(edges)
naive_model.fit(train_df, estimator=DiscreteBayesianEstimator())

print(naive_model.edges())

[('pass', 'school'), ('pass', 'sex'), ('pass', 'age'), ('pass', 'address'), ('pass', 'famsize'), ('pass', 'Pstatus'), ('pass', 'Medu'), ('pass', 'Fedu'), ('pass', 'Mjob'), ('pass', 'Fjob'), ('pass', 'reason'), ('pass', 'guardian'), ('pass', 'traveltime'), ('pass', 'studytime'), ('pass', 'failures'), ('pass', 'schoolsup'), ('pass', 'famsup'), ('pass', 'paid'), ('pass', 'activities'), ('pass', 'nursery'), ('pass', 'higher'), ('pass', 'internet'), ('pass', 'romantic'), ('pass', 'famrel'), ('pass', 'freetime'), ('pass', 'goout'), ('pass', 'Dalc'), ('pass', 'Walc'), ('pass', 'health'), ('pass', 'absences')]


In [ ]:
evaluate_model(naive_model, X_test, y_test)

100%|██████████| 79/79 [00:00<00:00, 81.43it/s] 


ValueError: Found input variables with inconsistent numbers of samples: [316, 79]

# Model 2: Tree Augmented Naive Bayes (TAN)

In [ ]:
train_df = X_train.copy()
train_df['pass'] = y_train.values.astype(str)

tan = TreeSearch(train_df)

g = tan.estimate(
    estimator_type='tan',
    class_node='pass'
)

tan_model = DiscreteBayesianNetwork(g.edges())
tan_model.fit(train_df, estimator=DiscreteBayesianEstimator())

print(tan_model.edges())

Building tree: 100%|██████████| 465/465.0 [00:00<00:00, 2349.50it/s]
Building tree: 100%|██████████| 465/465.0 [00:00<00:00, 1369.60it/s]


[('Medu', 'Mjob'), ('Medu', 'Fedu'), ('Medu', 'famsup'), ('Mjob', 'freetime'), ('Mjob', 'internet'), ('Fedu', 'Fjob'), ('Fedu', 'nursery'), ('famsup', 'paid'), ('freetime', 'goout'), ('freetime', 'failures'), ('freetime', 'Pstatus'), ('goout', 'Walc'), ('goout', 'health'), ('goout', 'absences'), ('failures', 'guardian'), ('failures', 'higher'), ('Walc', 'Dalc'), ('Walc', 'studytime'), ('Walc', 'famrel'), ('Walc', 'traveltime'), ('Walc', 'sex'), ('absences', 'romantic'), ('guardian', 'age'), ('Dalc', 'activities'), ('Dalc', 'famsize'), ('studytime', 'reason'), ('studytime', 'schoolsup'), ('traveltime', 'address'), ('traveltime', 'school'), ('pass', 'school'), ('pass', 'sex'), ('pass', 'age'), ('pass', 'address'), ('pass', 'famsize'), ('pass', 'Pstatus'), ('pass', 'Medu'), ('pass', 'Fedu'), ('pass', 'Mjob'), ('pass', 'Fjob'), ('pass', 'reason'), ('pass', 'guardian'), ('pass', 'traveltime'), ('pass', 'studytime'), ('pass', 'failures'), ('pass', 'schoolsup'), ('pass', 'famsup'), ('pass', '

In [ ]:
evaluate_model(tan_model, X_test, y_test)

  0%|          | 0/79 [00:00<?, ?it/s]

100%|██████████| 79/79 [00:00<00:00, 92.09it/s] 


Accuracy: 0.60759
              precision    recall  f1-score   support

           0       0.38      0.31      0.34        26
           1       0.69      0.75      0.72        53

    accuracy                           0.61        79
   macro avg       0.54      0.53      0.53        79
weighted avg       0.59      0.61      0.60        79



# Model 3: Learned Bayesian Network

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import ExhaustiveSearch, MaximumLikelihoodEstimator

df_model = X.copy()

# structure learning
es = ExhaustiveSearch(df_model)
best_model = es.estimate()

# build BN
tan_model = DiscreteBayesianNetwork(best_model.edges())

# parameter learning
tan_model.fit(df_model, estimator=MaximumLikelihoodEstimator)

KeyboardInterrupt: 

# Experiment

# Results